# CD CosMx WTX biopsies: Treg counts and Treg labels

1. From `norm_anno.h5ad` (`01`), group cells as Treg, myeloid, connective or other, and export the raw counts of all 1,204 AUCell Tregs (`count_csv/all_treg_raw_counts.csv`, query of the Treg label transfer in `09_treg_label_transfer.R`).
2. Score Treg gene panels on the normalised Treg counts and keep core Tregs (core Treg score ≥ 0, cell-cycle score < 4; 494 cells).
3. Store `cell_type_treg` (core vs other Tregs) and `cell_type_treg_cci` per cell and write the biopsy object with Treg labels.

**Output:** `count_csv/all_treg_raw_counts.csv`; `h5ad_obj/norm_anno_tregbin.h5ad` (input of `10` and `11`).

The last part writes the LIANA ligand-receptor object of the inflamed biopsy slide (`h5ad_obj/lr_liana_i0476_1_nn_db.h5ad`, input of `10`).

In [ ]:
import scanpy as sc
import pandas as pd

In [ ]:
adata=sc.read_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/h5ad_obj/norm_anno.h5ad')

In [ ]:
treg_others = []
for i,j in adata.obs.iterrows():
    cell_cat = j['cell_category']
    cell_type = j['cell_type_short']
    if cell_type == 'Treg':
        treg_others.append('Treg')
    elif cell_cat=='Myeloid':
        treg_others.append('Myeloid')
    elif cell_cat=='Connective':
        treg_others.append('Connective')
    else:
        treg_others.append('Others')

In [ ]:
adata.obs['treg_others']=treg_others

In [ ]:
tregs = adata[adata.obs['cell_type']=='Treg']

In [ ]:
tregs.X = tregs.layers['raw']

In [ ]:
sc.pp.normalize_total(tregs, target_sum=1e4)   # library-size normalize

In [ ]:
raw_counts = pd.DataFrame(tregs.layers['raw'].toarray())
raw_counts.columns = tregs.var_names.tolist()
raw_counts.index = tregs.obs['cell'].tolist()

In [ ]:
raw_counts.to_csv('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/count_csv/all_treg_raw_counts.csv')

In [ ]:
panels = {
 'Activated_tissue': ['BATF','PRDM1','MAF','TNFRSF18','TNFRSF4'],
 'Tfr': ['BCL6','CXCR5','PDCD1','ICOS'],
 'Th17_like': ['RORC','CCR6','IL17A','IL17F'],
 'IFN_response': ['ISG15','IFI6','MX1','IFIT3'],
 'Cell_cycle': ['MKI67','TOP2A','STMN1'],
 'Treg_core':['FOXP3','IL2RA','CTLA4','TIGIT','ENTPD1','LRRC32','IL10']
}
for name, genes in panels.items():
    sc.tl.score_genes(tregs, [g for g in genes if g in tregs.var_names], score_name=name)

In [ ]:
treg_strict = tregs[tregs.obs["Treg_core"] >= 0].copy()

In [ ]:
treg_strict = treg_strict[treg_strict.obs["Cell_cycle"] < 4].copy()

In [ ]:
clean_tregs =treg_strict.obs['cell'].tolist()

In [ ]:
cell_type_treg = []
for i,j in adata.obs.iterrows():
    cell_type = j['cell_type_short']
    if cell_type == 'Treg':
        if j['cell'] in clean_tregs:
            cell_type_treg.append('Core Treg')
        else:
            cell_type_treg.append('NC Treg')
    else:
        cell_type_treg.append(cell_type)

In [ ]:
adata.obs['cell_type_treg']=cell_type_treg

In [ ]:
cell_type_treg_cci = []
for i,j in zip(adata.obs['cell_type_treg'].tolist(),adata.obs['cell_category'].tolist()):
    if j == 'T':
        if i == 'Core Treg':
            cell_type_treg_cci.append (i)
        else:
            cell_type_treg_cci.append ('Other T')
    else:
        cell_type_treg_cci.append(j)

In [ ]:
adata.obs['cell_type_treg_cci']=cell_type_treg_cci

In [ ]:
adata.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/h5ad_obj/norm_anno_tregbin.h5ad')

## LIANA bivariate ligand-receptor scores (WTX biopsy, slide i0476_slide1)

Per-cell LIANA bivariate scores (cosine local score, global Moran's I, 100 permutations) on slide `i0476_slide1` (inflamed I0476), with the NicheNet ligand-receptor network (`lr_network_human_21122021.rds`, Zenodo record 7074291) as resource. Spatial weights: Gaussian kernel, bandwidth 300, cutoff 0.1, up to 500 neighbours, on the FOV coordinates with the y axis flipped. `10_treg_suppression_score.py` reads the pair-level Moran's I of this object to order the antigen-presentation LR module.

In [ ]:
import liana as li
import pyreadr

In [ ]:
adata=sc.read_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/h5ad_obj/norm_anno_tregbin.h5ad')

In [ ]:
adata_i0476_1 = adata[adata.obs['slide']=='i0476_slide1']

In [ ]:
# Copy spatial_fov to spatial, but flip the Y-axis
coords = adata_i0476_1.obsm['spatial_fov'].copy()

# Flip Y coordinates
y_max = coords[:, 1].max()
y_min = coords[:, 1].min()
coords[:, 1] = y_max + y_min - coords[:, 1]  # Mirror around center

# Set as spatial
adata_i0476_1.obsm['spatial'] = coords

In [ ]:
li.ut.spatial_neighbors(adata_i0476_1, bandwidth=300, cutoff=0.1, set_diag=True, max_neighbours=500)

In [ ]:
import requests
url = "https://zenodo.org/record/7074291/files/lr_network_human_21122021.rds"
r = requests.get(url)
with open("/share/fsmresfiles/ydn4687/nichenet/lr_network_human_21122021.rds", "wb") as f:
    f.write(r.content)



In [ ]:
# Now read it with pyreadr
lr = pyreadr.read_r("/share/fsmresfiles/ydn4687/nichenet/lr_network_human_21122021.rds")


In [ ]:
lr_df = next(iter(lr.values()))

In [ ]:
lr_df=lr_df[['from','to']]

In [ ]:
lr_df.columns = ['ligand','receptor']

In [ ]:
lrdata_nn = li.mt.bivariate(adata_i0476_1,
                resource=lr_df, # NOTE: uses HUMAN gene symbols!
                local_name='cosine', # Name of the function
                global_name="morans", # Name global function
                n_perms=100, # Number of permutations to calculate a p-value
                mask_negatives=False, # Whether to mask LowLow/NegativeNegative interactions
                add_categories=True, # Whether to add local categories to the results
                nz_prop=0.05, # Minimum expr. proportion for ligands/receptors and their subunits
                use_raw=False,
                verbose=True
                )

In [ ]:
lrdata_nn.write_h5ad('/share/fsmresfiles/UC/AtoMx/CD_6k_wholetrans/whole_trans/Processed_merged/h5ad_obj/lr_liana_i0476_1_nn_db.h5ad')